In [ ]:
# Cellule 1 : Vérification du GPU disponible sur Colab
!nvidia-smi

# Cellule 2 : Installation et imports
import time
import math
import numpy as np
from numba import cuda
from numba.cuda.random import create_xoroshiro128p_states, xoroshiro128p_normal_float64

@cuda.jit
def monte_carlo_option_kernel(rng_states, S0, K, T, r, sigma, N_per_thread, d_payoffs):
    """
    Noyau CUDA : chaque thread simule N_per_thread trajectoires
    du Mouvement Brownien Géométrique et accumule les payoffs actualisés.
    """
    idx = cuda.grid(1)
    if idx < d_payoffs.shape[0]:
        accumulated_payoff = 0.0
        discount_factor = math.exp(-r * T)
        drift = (r - 0.5 * sigma * sigma) * T
        vol_sqrtT = sigma * math.sqrt(T)

        for _ in range(N_per_thread):
            # Génération d'une variable normale centrée réduite ~ N(0, 1)
            z = xoroshiro128p_normal_float64(rng_states, idx)
            ST = S0 * math.exp(drift + vol_sqrtT * z)
            payoff = max(ST - K, 0.0)
            accumulated_payoff += payoff

        d_payoffs[idx] = (accumulated_payoff / N_per_thread) * discount_factor


# Cellule 3 : Lancement sur VRAI GPU à grande échelle (N = 100 Millions)
N_TOTAL = 100_000_000
threads_per_block = 256
blocks = 1024
total_threads = blocks * threads_per_block
N_per_thread = math.ceil(N_TOTAL / total_threads)

print(f"Lancement de {N_TOTAL:,} simulations sur GPU réel...")

# Mesure du temps
start_time = time.time()

rng_states = create_xoroshiro128p_states(total_threads, seed=1234)
d_payoffs = cuda.device_array(total_threads, dtype=np.float64)

# Exécution du noyau GPU
monte_carlo_option_kernel[blocks, threads_per_block](
    rng_states, 100.0, 100.0, 1.0, 0.05, 0.20, N_per_thread, d_payoffs
)
cuda.synchronize()

gpu_time = time.time() - start_time
final_price = float(np.mean(d_payoffs.copy_to_host()))

print(f"Prix calculé : {final_price:.4f} €")
print(f"Temps GPU total : {gpu_time:.4f} secondes")
print(f"Débit : {N_TOTAL / gpu_time / 1e6:.2f} Millions de simulations / seconde")

zsh:1: command not found: nvidia-smi
Cible visée : 100,000,000 simulations
N réel exécuté : 100,139,008 simulations (via 262144 threads)

Échauffement du kernel (compilation JIT en cours)...


/home/darwin/Documents/Tafs/Python/School/GPGPU/Monte-carlo-project/.venv/lib/python3.12/site-packages/numba/cuda/random.py:225: NumbaWarning: 
Compilation is falling back to object mode WITHOUT looplifting enabled because Function "init_xoroshiro128p_states_cpu" failed type inference due to: Invalid use of type(CPUDispatcher(<function init_xoroshiro128p_state at 0x776fe7c46ca0>)) with parameters (array(Record(s0[type=uint64;offset=0],s1[type=uint64;offset=8];16;True), 1d, C), Literal[int](0), uint64)

During: resolving callee type: type(CPUDispatcher(<function init_xoroshiro128p_state at 0x776fe7c46ca0>))
During: typing of call at /home/darwin/Documents/Tafs/Python/School/GPGPU/Monte-carlo-project/.venv/lib/python3.12/site-packages/numba/cuda/random.py (232)

File "../../.venv/lib/python3.12/site-packages/numba/cuda/random.py", line 232:
def init_xoroshiro128p_states_cpu(states, seed, subsequence_start):
    <source elided>
    if n >= 1:
        init_xoroshiro128p_state(states, 0, se

KeyboardInterrupt: 